# StyleVAR: SFT vs GRPO Comparison
Compare SFT (Output_v2) and GRPO (grpo_output_v2) checkpoints side-by-side.
Saves individual images + source images to `samples/` directory.

In [ ]:
import os, sys, glob, math, random, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import numpy as np

ROOT = os.path.dirname(os.path.abspath('__file__'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- Config ----
NUM_SAMPLES = 8
SEED = 42
TOP_K, TOP_P = 900, 0.96

SFT_CKPT_DIR = os.path.join(ROOT, 'Output_v2')          # SFT v2 checkpoint dir
GRPO_CKPT_DIR = os.path.join(ROOT, 'grpo_output_v2')     # GRPO checkpoint dir
SAMPLE_DIR = os.path.join(ROOT, 'samples')                # output dir for saved images

os.makedirs(SAMPLE_DIR, exist_ok=True)
print(f'Device: {device}')
print(f'SFT dir:  {SFT_CKPT_DIR}')
print(f'GRPO dir: {GRPO_CKPT_DIR}')
print(f'Output:   {SAMPLE_DIR}')

## 1. Helpers

In [ ]:
from models import VQVAE, StyleVAR, build_vae_stylevar

# ---- LoRA (for loading GRPO checkpoints) ----
class LoRALinear(nn.Module):
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.base_weight = base_linear.weight
        self.base_weight.requires_grad_(False)
        self.has_bias = base_linear.bias is not None
        if self.has_bias:
            self.base_bias = base_linear.bias
            self.base_bias.requires_grad_(False)
        dev, dtype = base_linear.weight.device, base_linear.weight.dtype
        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=dev, dtype=dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=dev, dtype=dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scaling = alpha / rank
        self._enabled = True

    @property
    def weight(self):
        if self._enabled:
            return self.base_weight + (self.lora_B @ self.lora_A) * self.scaling
        return self.base_weight

    @property
    def bias(self):
        return self.base_bias if self.has_bias else None

    def forward(self, x):
        return F.linear(x, self.weight, self.bias)

def apply_lora(model, rank, alpha):
    for block in model.blocks:
        for attr in ("mat_qkv_guide", "mat_qkv_target", "proj"):
            setattr(block.attn, attr, LoRALinear(getattr(block.attn, attr), rank, alpha))
        for attr in ("fc1", "fc2"):
            setattr(block.ffn, attr, LoRALinear(getattr(block.ffn, attr), rank, alpha))

def set_lora_enabled(model, enabled):
    for block in model.blocks:
        for attr in ("mat_qkv_guide", "mat_qkv_target", "proj"):
            m = getattr(block.attn, attr)
            if isinstance(m, LoRALinear): m._enabled = enabled
        for attr in ("fc1", "fc2"):
            m = getattr(block.ffn, attr)
            if isinstance(m, LoRALinear): m._enabled = enabled

# ---- Image helpers ----
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

def load_img(path):
    return transform(Image.open(path).convert('RGB')).unsqueeze(0).to(device)

def tensor_to_pil(t):
    """(3,H,W) in [0,1] -> PIL"""
    return Image.fromarray((t.clamp(0,1).permute(1,2,0).cpu().numpy() * 255).astype('uint8'))

def load_sft_state(ckpt_path):
    """Load SFT checkpoint, handle wrapped format."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if 'trainer' in ckpt and 'var_wo_ddp' in ckpt['trainer']:
        return ckpt['trainer']['var_wo_ddp']
    elif 'model' in ckpt:
        return ckpt['model']
    return ckpt

print('Helpers ready.')

## 2. Build model + Load SFT & GRPO

In [ ]:
# ---- Build VAE + StyleVAR ----
vae, model = build_vae_stylevar(
    V=4096, Cvae=32, ch=160, share_quant_resi=4, device=device,
    patch_nums=(1,2,3,4,5,6,8,10,13,16),
    depth=20, shared_aln=False, attn_l2_norm=True,
    flash_if_available=True, fused_if_available=True,
    init_adaln=0.5, init_adaln_gamma=1e-5, init_head=0.02, init_std=-1,
    style_enc_dim=512,
)
vae.load_state_dict(torch.load(os.path.join(ROOT, 'ckpt', 'vae_ch160v4096z32.pth'),
                                map_location='cpu', weights_only=False), strict=True)
vae.eval()
print('VAE loaded.')

# ---- Find checkpoints ----
sft_ckpt = os.path.join(SFT_CKPT_DIR, 'ar-ckpt-best.pth')
assert os.path.exists(sft_ckpt), f'SFT ckpt not found: {sft_ckpt}'

grpo_ckpt = os.path.join(GRPO_CKPT_DIR, 'grpo_best.pth')
if not os.path.exists(grpo_ckpt):
    # fallback to latest
    grpo_ckpt = os.path.join(GRPO_CKPT_DIR, 'grpo_latest.pth')
has_grpo = os.path.exists(grpo_ckpt)

print(f'SFT ckpt:  {sft_ckpt}')
print(f'GRPO ckpt: {grpo_ckpt} (exists={has_grpo})')

# ---- Load SFT base into model ----
sft_state = load_sft_state(sft_ckpt)
model.load_state_dict(sft_state, strict=True)
model.eval()
print('SFT model loaded.')

# ---- Inject LoRA + load GRPO weights ----
if has_grpo:
    grpo_raw = torch.load(grpo_ckpt, map_location='cpu', weights_only=False)
    grpo_args = grpo_raw.get('args', {})
    rank = grpo_args.get('lora_rank', 256)
    alpha = grpo_args.get('lora_alpha', 512.0)
    grpo_step = grpo_raw.get('step', '?')

    apply_lora(model, rank, alpha)
    sd = model.state_dict()
    sd.update(grpo_raw['model'])
    model.load_state_dict(sd)
    model.eval()
    print(f'GRPO LoRA loaded (step={grpo_step}, rank={rank})')
else:
    print('No GRPO checkpoint found — will only show SFT results.')

## 3. Sample data & Generate

In [ ]:
random.seed(SEED)

samples = []  # list of (content_path, style_path)

# ---- 4 from ImagePulse ----
ip_root = os.path.join(ROOT, 'data', 'ImagePulse')
all_ip_dirs = sorted([d for d in os.listdir(ip_root)
                      if os.path.isdir(os.path.join(ip_root, d))])
random.shuffle(all_ip_dirs)
for sd in all_ip_dirs:
    c = os.path.join(ip_root, sd, 'content.png')
    s = os.path.join(ip_root, sd, 'style.png')
    if os.path.isfile(c) and os.path.isfile(s):
        samples.append((c, s))
    if len(samples) >= 4:
        break
print(f'ImagePulse: {len(samples)} samples')

# ---- 4 from OmniStyle ----
omni_root = os.path.join(ROOT, 'data', 'OmniStyle-150k')
omni_content_dir = os.path.join(omni_root, 'content')
omni_style_dir = os.path.join(omni_root, 'style')
omni_target_dir = None
for cand in ('target', 'OmniStyle-150K', 'OmniStyle-150k'):
    p = os.path.join(omni_root, cand)
    if os.path.isdir(p) or os.path.islink(p):
        omni_target_dir = p
        break

if omni_target_dir:
    all_target_files = [f for f in os.listdir(omni_target_dir) if '&&' in f]
    random.shuffle(all_target_files)
    omni_count = 0
    for tf in all_target_files:
        try:
            c_name, s_raw = tf.split('&&')
            s_name = s_raw[:-4]  # strip .png suffix from target filename
            c_path = os.path.join(omni_content_dir, c_name)
            s_path = os.path.join(omni_style_dir, s_name)
            if os.path.isfile(c_path) and os.path.isfile(s_path):
                samples.append((c_path, s_path))
                omni_count += 1
        except Exception:
            continue
        if omni_count >= 4:
            break
    print(f'OmniStyle:  {omni_count} samples')
else:
    print(f'OmniStyle target dir not found at {omni_root}')

print(f'Total: {len(samples)} samples')

# ---- Generate with both models ----
sft_results = []
grpo_results = []

with torch.no_grad():
    for i, (c_path, s_path) in enumerate(samples):
        content = load_img(c_path)
        style = load_img(s_path)

        # SFT: disable LoRA
        if has_grpo:
            set_lora_enabled(model, False)
        gen_sft = model.autoregressive_infer(
            B=1, style_img=style, content_img=content,
            top_k=TOP_K, top_p=TOP_P, g_seed=SEED + i,
        )
        sft_results.append(tensor_to_pil(gen_sft[0]))

        # GRPO: enable LoRA
        if has_grpo:
            set_lora_enabled(model, True)
            gen_grpo = model.autoregressive_infer(
                B=1, style_img=style, content_img=content,
                top_k=TOP_K, top_p=TOP_P, g_seed=SEED + i,
            )
            grpo_results.append(tensor_to_pil(gen_grpo[0]))

        src = 'ImagePulse' if i < 4 else 'OmniStyle'
        print(f'  [{i+1}/{len(samples)}] {src} done')

print(f'Generated {len(sft_results)} SFT + {len(grpo_results)} GRPO images')

## 4. Save individual images to `samples/`

In [ ]:
for i, (c_path, s_path) in enumerate(samples):
    sample_dir = os.path.join(SAMPLE_DIR, f'{i:03d}')
    os.makedirs(sample_dir, exist_ok=True)

    # Copy source images (original resolution)
    shutil.copy2(c_path, os.path.join(sample_dir, 'content.png'))
    shutil.copy2(s_path, os.path.join(sample_dir, 'style.png'))

    # Save SFT result
    sft_results[i].save(os.path.join(sample_dir, 'sft.png'))

    # Save GRPO result (if available)
    if grpo_results:
        grpo_results[i].save(os.path.join(sample_dir, 'grpo.png'))

    print(f'  [{i+1}/{len(samples)}] saved to {sample_dir}/')

print(f'\nAll saved to {SAMPLE_DIR}/')
print(f'Structure per sample:')
print(f'  samples/000/content.png  (original)')
print(f'  samples/000/style.png    (original)')
print(f'  samples/000/sft.png      (generated)')
if grpo_results:
    print(f'  samples/000/grpo.png     (generated)')

## 5. Visualization (Content | Style | SFT | GRPO)

In [ ]:
N = len(samples)
ncols = 4 if grpo_results else 3
fig, axes = plt.subplots(N, ncols, figsize=(4 * ncols, 4 * N))
if N == 1:
    axes = axes[None, :]

titles = ['Content', 'Style', 'SFT', 'GRPO'] if grpo_results else ['Content', 'Style', 'SFT']
for j, t in enumerate(titles):
    axes[0, j].set_title(t, fontsize=14, fontweight='bold')

for i, (c_path, s_path) in enumerate(samples):
    # Load originals at 256x256 for display
    c_img = Image.open(c_path).convert('RGB').resize((256, 256))
    s_img = Image.open(s_path).convert('RGB').resize((256, 256))

    axes[i, 0].imshow(c_img)
    axes[i, 1].imshow(s_img)
    axes[i, 2].imshow(sft_results[i])
    if grpo_results:
        axes[i, 3].imshow(grpo_results[i])

    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
save_path = os.path.join(SAMPLE_DIR, 'comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Comparison grid saved to {save_path}')